# Every way to specify a systematic with `graphed.vary`

A guided tour, simplest to most complex. Every cell is executed and prints the
`graphed.labels()` / `graphed.points()` / `graphed.variations()` it actually produces — no number
or label below is asserted from memory.

The notebook is self-contained: toy numpy/awkward arrays and a tiny inline correctionlib set built
in the next cell. One level (the process-pool proof) also reads the 50k skim committed in
`data/`, purely for realism.

> **Companion — the standard path.** The [ADL benchmark](./graphed-adl-benchmarks.ipynb) runs the same JES + b-tag setup on the real 50k skim the plain way: one `graphed.vary` per nuisance, propagation for free (level 9), and the seven-template one-at-a-time union a datacard wants — no `points=`. This tour is the deep dive into the non-standard profiles (joint universes, scale grids, PDF sets) that benchmark points here for.

## The map

An analyst makes three orthogonal choices, and the tour is their product plus the three ways
universes can relate.

**(a) What is varied.** A bare `Array` (the *loose* form), an event context's **weight**
(`is_weight=True` + `variations=`), or an event context's **collections** (`collections=`). All
three mint the same label grammar `f"{name}_{tag}"` and the same default point `{name: tag}`.

**(b) How many nuisance families.** One call; a second call extending the *same* family; or
independent families, which compose as the **union**, never the cross product.

**(c) How universes relate.** Three genuinely distinct mechanisms, which the word "correlated"
conflates and which this notebook teaches apart:

| Analyst intent | Mechanism | Level |
|---|---|---|
| Two registrations are the same fit parameter | **name identity** — share the nuisance `name` | 8 |
| A correction is a function of a quantity a nuisance moves | **propagation** — compute it from the varied quantity | 9 |
| A universe displaced on ≥2 axes at once | **`points=`** — register the universe with its coordinate map | 10+ |

## Vocabulary

- **nuisance** — a family name; the parameter the fit sees.
- **coordinate** — the per-axis displacement (HistFactory's α, combine's ν).
- **point** — a sparse `{nuisance: coordinate}` map. HS3 calls this a *parameter point*.
- **correlated** — shares a nuisance name. Nothing else.
- **propagation** — a correction depends on a quantity a nuisance moves. *Not* correlation.
- **one-at-a-time set** — the axis-aligned unit points; what a datacard normally wants.
- **factorization error** — what a joint universe measures.

**Every universe is a point in nuisance space; a label is a NAME for that point; resolution
projects the requested point onto the axes a container knows and then falls back to nominal.**
`nominal` is the origin: every coordinate at 0.

## Level 0 — setup

Two toy datasets and one toy correctionlib set. Nothing else is imported.

- `ctx0()` — a numpy-backed event context with two collections, `pt` and `eta`. Used wherever the
  level is about *declaration shape* rather than physics.
- `toy_jets()` — jagged jet pT, an awkward array, used where a correction has to be evaluated.
- `TOY_SF` — a correctionlib v2 set whose `up`/`down` uncertainty **grows with pT**
  (2% / 5% / 10% / 20% in the pT bins `[0,30,60,100,∞)`). That pT dependence is one of the two
  sources of the factorization error measured in level 12; a scale factor flat in pT, read on a
  selection frozen at nominal, makes that error read zero — level 12 runs exactly that leg as its
  control.

In [1]:
import json

import awkward as ak
import correctionlib
import numpy as np

import graphed
from graphed import Session
from graphed.awkward import AwkwardBackend, from_awkward, gak
from graphed.context import EventContext
from graphed.numpy import NumpyBackend, from_record


def ctx0():
    """A fresh Session + a 12-event context with `pt` and `eta` collections."""
    s = Session(NumpyBackend())
    r = from_record(s, "ev", pt=np.arange(1.0, 13.0), eta=np.arange(1.0, 13.0) / 10.0)
    return s, EventContext(s, r["pt"], collections={"pt": r["pt"], "eta": r["eta"]})


def toy_jets(n_events=2000, seed=7):
    """Jagged jet pT: 2-5 jets per event, exponential pT, straddling the SF bin edges."""
    rng = np.random.default_rng(seed)
    counts = rng.integers(2, 6, size=n_events)
    pt = rng.exponential(45.0, size=int(counts.sum())) + 10.0
    return ak.unflatten(pt, counts)


def toy_session():
    s = Session(AwkwardBackend())
    return s, from_awkward(s, "jet_pt", toy_jets())


TOY_SF = json.dumps({
    "schema_version": 2,
    "description": "a pT-binned b-tag-like scale factor; the uncertainty grows with pT",
    "corrections": [{
        "name": "toy_sf",
        "version": 1,
        "inputs": [{"name": "systematic", "type": "string"}, {"name": "pt", "type": "real"}],
        "output": {"name": "sf", "type": "real"},
        "data": {
            "nodetype": "category", "input": "systematic",
            "content": [
                {"key": "central", "value": {"nodetype": "binning", "input": "pt",
                    "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                    "content": [1.00, 1.00, 1.00, 1.00], "flow": "clamp"}},
                {"key": "up", "value": {"nodetype": "binning", "input": "pt",
                    "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                    "content": [1.02, 1.05, 1.10, 1.20], "flow": "clamp"}},
                {"key": "down", "value": {"nodetype": "binning", "input": "pt",
                    "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                    "content": [0.98, 0.95, 0.90, 0.80], "flow": "clamp"}},
            ],
        },
    }],
}).encode()

SF = correctionlib.CorrectionSet.from_string(TOY_SF.decode())["toy_sf"]


def show(container, *, variations=False):
    """labels(), then points() one per line, then optionally variations()."""
    print("labels    :", graphed.labels(container))
    pts = graphed.points(container)
    print("points    :")
    for label in graphed.labels(container):
        print(f"    {label:22s} {pts[label]}")
    if variations:
        print("variations:", graphed.variations(container))


print("correctionlib", correctionlib.__version__, "| toy SF at pT=25/50/80/200, systematic=up:",
      [round(SF.evaluate("up", p), 2) for p in (25.0, 50.0, 80.0, 200.0)])

correctionlib 2.9.0 | toy SF at pT=25/50/80/200, systematic=up: [1.02, 1.05, 1.1, 1.2]


## Level 1 — one at a time, on a bare array

The simplest systematic: two keyword tags on an `Array`. The label is `f"{name}_{tag}"` and its
point is the axis-aligned `{name: tag}` — every other nuisance sits at 0, which is what absence
means. `nominal` maps to the empty point, the origin.

This is the *loose* form: no event context, no weight, just a varied quantity.

In [2]:
s, c = ctx0()
pt = c["pt"]

jes = graphed.vary(pt, "jes", up=pt * 1.1, down=pt * 0.9)
show(jes)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}


## Level 2 — extending a family

A nuisance family is open. A second `graphed.vary` call with the *same* name adds tags to it, so
the fit still sees **one** parameter. Three tags on one axis, not three axes.

In [3]:
jes2 = graphed.vary(jes, "jes", up2=graphed.nominal(jes) * 1.21)
show(jes2)

labels    : ('nominal', 'jes_up', 'jes_down', 'jes_up2')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    jes_up2                {'jes': 'up2'}


## Level 3 — a weight systematic

`is_weight=True` on an event context: the selection and the observable are identical in every
universe, only the per-event weight moves. `graphed.variations()` reports the kind word
`'weight'`.

In [4]:
s, c = ctx0()
w = c["pt"] * 0.5

btag = graphed.vary(c, "btag", w, is_weight=True,
                    variations={"up": w * 1.2, "down": w * 0.8})
show(btag, variations=True)

labels    : ('nominal', 'btag_up', 'btag_down')
points    :
    nominal                {}
    btag_up                {'btag': 'up'}
    btag_down              {'btag': 'down'}
variations: {'btag': {'up': ('weight', None), 'down': ('weight', None)}}


## Level 4 — a shift systematic

`collections=` instead: the *kinematics* move, so the selection and the observable both change.
Same declaration shape, same label grammar, same default points — but `graphed.variations()`
reports `'shift'`, and downstream every cut is re-evaluated per universe.

In [5]:
s, c = ctx0()
pt = c["pt"]

shift = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
show(shift, variations=True)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
variations: {'jes': {'up': ('shift', None), 'down': ('shift', None)}}


## Level 5 — lockstep: one nuisance, two collections

One nuisance can move several collections **coherently** — a jet-energy scale that must move `pt`
and `eta` together. The universe count does **not** grow: still three labels, because it is still
one axis.

In [6]:
s, c = ctx0()
pt, eta = c["pt"], c["eta"]

lock = graphed.vary(c, "jes", collections={
    "pt":  {"up": pt * 1.1,   "down": pt * 0.9},
    "eta": {"up": eta * 1.01, "down": eta * 0.99},
})
show(lock)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}


## Level 6 — stacked independent families compose as the UNION

Two independent 2-tag families give **5** universes (`1 + 2 + 2`), not 9. A label registered
without `points=` differs from nominal on exactly **one** axis, so no cross product can arise
implicitly. This axis-aligned set is the one-at-a-time set the datacard wants.

In [7]:
ambient = graphed.weight(btag)          # the btag family from level 3
stacked = graphed.vary(btag, "mu", ambient, is_weight=True,
                       variations={"up": ambient * 1.05, "down": ambient * 0.95})
show(stacked, variations=True)
print()
print("2 tags + 2 tags ->", len(graphed.labels(stacked)), "universes, not", 3 * 3)

labels    : ('nominal', 'btag_up', 'btag_down', 'mu_up', 'mu_down')
points    :
    nominal                {}
    btag_up                {'btag': 'up'}
    btag_down              {'btag': 'down'}
    mu_up                  {'mu': 'up'}
    mu_down                {'mu': 'down'}
variations: {'btag': {'up': ('weight', None), 'down': ('weight', None)}, 'mu': {'up': ('weight', None), 'down': ('weight', None)}}

2 tags + 2 tags -> 5 universes, not 9


## Level 7 — shift then weight

A kinematic family and a weight family stack the same way. Note the second line: the **ambient
weight** carries the inherited `jes` labels even though `jes` is not one of *its* registered
families — a container's labels legitimately outrun its own tag map, which is why axis sets are
read from the Session registry rather than derived from the tags.

In [8]:
w7 = shift["pt"] * 0.5                  # the shift context from level 4
both = graphed.vary(shift, "btag", w7, is_weight=True,
                    variations={"up": w7 * 1.2, "down": w7 * 0.8})

print("context labels:", graphed.labels(both))
print("weight  labels:", graphed.labels(graphed.weight(both)))
print("variations    :", graphed.variations(both))

context labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down')
weight  labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down')
variations    : {'btag': {'up': ('weight', None), 'down': ('weight', None)}, 'jes': {'up': ('shift', None), 'down': ('shift', None)}}


## Level 8 — mechanism 1: NAME IDENTITY

The first correlation mechanism. Registering **one nuisance name** as both a shift and a weight
ties them into a single fit parameter — combine's rule verbatim: *"multiple instances of any
nuisance parameter, sharing the same name, are treated as a single parameter."*

The label set does **not** grow: one universe carries both effects. `graphed.variations()` reports
the third kind word `'both'` for the dual tag, while the shift-only tag stays `'shift'`.

In [9]:
s, c = ctx0()
pt = c["pt"]

sh = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
w8 = sh["pt"] * 0.5
dual = graphed.vary(sh, "jes", w8, is_weight=True, variations={"up": w8 * 1.3})

print("labels    :", graphed.labels(dual), "  <- still three; jes_up now moves BOTH")
print("variations:", graphed.variations(dual))

labels    : ('nominal', 'jes_up', 'jes_down')   <- still three; jes_up now moves BOTH
variations: {'jes': {'up': ('both', None), 'down': ('shift', None)}}


## Level 9 — mechanism 2: PROPAGATION

The second mechanism, and the one most often mislabelled "correlation". The toy scale factor is a
**function of jet pT**, and the `jes` nuisance moves pT. Inside the `jes_up` universe the scale
factor must be re-evaluated on the *shifted* jets.

`gak.apply_correction` is container-traversing: hand it the **varied** pT and it returns a `Varied`
over the same labels, with a **distinct node per universe** — the correction genuinely re-evaluated
on each universe's own jets. Nothing declares this; it falls out of passing the varied quantity.

(The older workaround of fanning out by hand over universes is obsolete: `apply_correction`'s plan
now pickles and crosses a process pool, which level 13 proves.)

In [10]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)

sf_central = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                  args=["central", "$0"])

print("type:", type(sf_central).__name__, " labels:", graphed.labels(sf_central))
for label in graphed.labels(sf_central):
    print(f"    {label:10s} -> IR node {graphed.universe(sf_central, label).node_id}")
print()
print("propagation is not correlation: `jes` is the only registered nuisance;")
print("the SF is simply a function of a quantity `jes` moves.")

type: Varied  labels: ('nominal', 'jes_up', 'jes_down')
    nominal    -> IR node 3
    jes_up     -> IR node 4
    jes_down   -> IR node 5

propagation is not correlation: `jes` is the only registered nuisance;
the SF is simply a function of a quantity `jes` moves.


## Level 10 — mechanism 3: `points=`, a universe on two axes at once

The third mechanism, and the only new one. A universe displaced on **≥2 axes at once** cannot be
named by a single `(family, tag)` pair, so `points=` attaches the coordinate map explicitly.

Rules, all visible below:

- `points=` is keyed by **tag**, exactly like `variations=`, and its keys must be tags registered
  in the same call.
- A tag absent from `points=` gets the **default** point `{name: tag}` — the implicit rule made
  explicit.
- A tag present in `points=` takes that point *instead of* the default; the family name is a name
  for the point, not an extra coordinate.
- The joint members are the **same expression objects** as the one-at-a-time members. The *point*,
  not a different expression, is what selects which inner `jes` universe each label reads.

Nine universes: nominal, 2 `jes`, 2 one-at-a-time `sf`, 4 joint. The label grammar is unchanged —
`sf_jesup_up` is an ordinary `name_tag` label; no point is ever auto-rendered into a label.

In [11]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)
keep = jes_pt > 30.0                       # a selection on the SHIFTED pT
passes = gak.sum(keep, axis=1) >= 2


def sf(systematic):
    """Per-jet SF off the VARIED pT (level 9), producted over the selected jets."""
    per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                   args=[systematic, "$0"])
    return gak.prod(per_jet[keep], axis=1) * passes


central, up, down = sf("central"), sf("up"), sf("down")

weight = graphed.vary(central, "sf",
    variations={"up": up, "down": down,                 # the one-at-a-time set
                "jesup_up": up, "jesup_down": down,     # ... and the four joint members,
                "jesdn_up": up, "jesdn_down": down},    #     the SAME objects
    points={"jesup_up":   {"jes": "up",   "sf": "up"},
            "jesup_down": {"jes": "up",   "sf": "down"},
            "jesdn_up":   {"jes": "down", "sf": "up"},
            "jesdn_down": {"jes": "down", "sf": "down"}})

show(weight)

labels    : ('nominal', 'jes_up', 'jes_down', 'sf_up', 'sf_down', 'sf_jesup_up', 'sf_jesup_down', 'sf_jesdn_up', 'sf_jesdn_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    sf_up                  {'sf': 'up'}
    sf_down                {'sf': 'down'}
    sf_jesup_up            {'jes': 'up', 'sf': 'up'}
    sf_jesup_down          {'jes': 'up', 'sf': 'down'}
    sf_jesdn_up            {'jes': 'down', 'sf': 'up'}
    sf_jesdn_down          {'jes': 'down', 'sf': 'down'}


## Level 11 — why a joint label executes: resolution by PROJECTION

A joint label is asked of containers that know only *some* of its axes. Each container contributes
its member at the **projection** of the point onto the axes it carries, then falls back to nominal.

Below, `obs` is the observable, which knows the `jes` axis only:

- `jes_up` → its own shifted member;
- `btag_up` → an axis it does not carry, so **nominal**;
- `jesbtag_upup` → the point `{jes: up, btag: up}` restricted to `{jes}` is `{jes: up}`, so it gets
  the **shifted** member.

The weight, which carries both axes, returns the registered joint value. This is the whole of the
resolution machinery, and it is why a joint universe costs only the interning of nodes that already
exist.

In [12]:
s, c = ctx0()
pt = c["pt"]
a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
wv = a["pt"] * 0.5
b = graphed.vary(a, "btag", wv, is_weight=True, variations={"up": wv * 1.2, "down": wv * 0.8})
amb = graphed.weight(b)

j = graphed.vary(b, "jesbtag", amb, is_weight=True, variations={"upup": amb * 1.37},
                 points={"upup": {"jes": "up", "btag": "up"}})

obs = j["pt"]                                  # knows the `jes` axis ONLY
for label in ("nominal", "jes_up", "btag_up", "jesbtag_upup"):
    m = graphed.member_of(obs, label)
    print(f"  obs[{label:14s}] node {m.node_id}  {[float(x) for x in list(s.materialize(m))[:3]]}")

print()
print("weight labels          :", graphed.labels(graphed.weight(j)))
print("weight[jesbtag_upup]   :",
      [float(x) for x in list(s.materialize(graphed.universe(graphed.weight(j), "jesbtag_upup")))[:3]])

  obs[nominal       ] node 1  [1.0, 2.0, 3.0]
  obs[jes_up        ] node 3  [1.1, 2.2, 3.3000000000000003]
  obs[btag_up       ] node 1  [1.0, 2.0, 3.0]
  obs[jesbtag_upup  ] node 3  [1.1, 2.2, 3.3000000000000003]

weight labels          : ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down', 'jesbtag_upup')
weight[jesbtag_upup]   : [0.3425, 1.37, 3.0825000000000005]


## Level 12 — what a joint universe MEASURES: the factorization error

A joint universe is **not** "covering the correlation" — the fit already correlates by name
(level 8), and HistFactory's response is a *sum of one-dimensional terms*
(`A = nominal + Σ_i I_i(θ_i)`) with no slot for a joint template. What a joint universe measures is
the **error** of that factorization: how far the true two-axis response sits from the linearized
prediction `jes_1D + sf_1D − nominal`.

Three legs, and it is their **ordering** that carries the lesson, not any absolute number:

| leg | SF pT-dependent? | selection on shifted pT? | expected |
|---|---|---|---|
| the teaching case | yes | yes | largest |
| migration only | no | yes | smaller, still nonzero |
| **positive control** | no | no | **0**, to machine precision |

The third leg is the control that proves the instrument reads zero when there is nothing to
measure. A demo built on a pT-*flat* scale factor with a frozen selection reads zero, and a joint
universe built that way would teach the opposite of the intended lesson.

In [13]:
def yields(pt_dependent, live_selection):
    s = Session(AwkwardBackend())
    raw = from_awkward(s, "jet_pt", toy_jets())
    jes_pt = graphed.vary(raw, "jes", up=raw * 1.05, down=raw * 0.95)
    keep = (jes_pt if live_selection else raw) > 30.0
    passes = gak.sum(keep, axis=1) >= 2

    def sf(systematic):
        arg = jes_pt if pt_dependent else (jes_pt * 0.0 + 50.0)   # freeze the SF's pT argument
        per_jet = gak.apply_correction(TOY_SF, "toy_sf", [arg], SF.evaluate,
                                       args=[systematic, "$0"])
        return gak.prod(per_jet[keep], axis=1) * passes

    central, up, down = sf("central"), sf("up"), sf("down")
    w = graphed.vary(central, "sf",
        variations={"up": up, "down": down, "jesup_up": up, "jesup_down": down,
                    "jesdn_up": up, "jesdn_down": down},
        points={"jesup_up":   {"jes": "up",   "sf": "up"},
                "jesup_down": {"jes": "up",   "sf": "down"},
                "jesdn_up":   {"jes": "down", "sf": "up"},
                "jesdn_down": {"jes": "down", "sf": "down"}})
    return {L: float(ak.sum(s.materialize(graphed.universe(w, L))))
            for L in graphed.labels(w)}


def factorization_error(title, tot):
    print(title)
    base = tot["nominal"]
    worst = 0.0
    for short, jes_label in (("jesup", "jes_up"), ("jesdn", "jes_down")):
        for tag in ("up", "down"):
            joint = tot[f"sf_{short}_{tag}"]
            linear = tot[jes_label] + tot[f"sf_{tag}"] - base       # the fit's 1-D sum
            err = joint - linear
            worst = max(worst, abs(err) / base)
            print(f"    {jes_label:8s} x sf_{tag:5s}: joint={joint:12.6f}  1D-sum={linear:12.6f}"
                  f"  error={err:+11.6f}  ({100 * err / base:+.4f}% of nominal)")
    print(f"    -> largest |error| = {100 * worst:.4f}% of nominal\n")
    return worst


w1 = factorization_error("pT-binned SF, live selection  (the teaching case)", yields(True, True))
w2 = factorization_error("CONTROL  pT-flat SF, live selection  (selection migration only)",
                         yields(False, True))
w3 = factorization_error("CONTROL  pT-flat SF, selection frozen at nominal  (must read 0)",
                         yields(False, False))

print(f"ordering holds: {w1 > w2 > w3}"
      f"   (pT-binned+live {100 * w1:.4f}%  >  migration-only {100 * w2:.4f}%  >  control {w3:.2e})")
print("control is zero to machine precision:", w3 < 1e-9)

pT-binned SF, live selection  (the teaching case)
    jes_up   x sf_up   : joint= 1974.875095  1D-sum= 1942.498495  error= +32.376601  (+2.2130% of nominal)
    jes_up   x sf_down : joint= 1127.367392  1D-sum= 1150.617130  error= -23.249738  (-1.5892% of nominal)
    jes_down x sf_up   : joint= 1828.447268  1D-sum= 1859.498495  error= -31.051226  (-2.1224% of nominal)
    jes_down x sf_down : joint= 1089.826382  1D-sum= 1067.617130  error= +22.209251  (+1.5181% of nominal)
    -> largest |error| = 2.2130% of nominal



CONTROL  pT-flat SF, live selection  (selection migration only)
    jes_up   x sf_up   : joint= 1730.679280  1D-sum= 1721.590460  error=  +9.088820  (+0.6212% of nominal)
    jes_up   x sf_down : joint= 1305.432745  1D-sum= 1313.285928  error=  -7.853183  (-0.5368% of nominal)
    jes_down x sf_up   : joint= 1629.570521  1D-sum= 1638.590460  error=  -9.019939  (-0.6165% of nominal)
    jes_down x sf_down : joint= 1238.010241  1D-sum= 1230.285928  error=  +7.724314  (+0.5280% of nominal)
    -> largest |error| = 0.6212% of nominal

CONTROL  pT-flat SF, selection frozen at nominal  (must read 0)
    jes_up   x sf_up   : joint= 1677.590460  1D-sum= 1677.590460  error=  +0.000000  (+0.0000% of nominal)
    jes_up   x sf_down : joint= 1269.285927  1D-sum= 1269.285928  error=  -0.000000  (-0.0000% of nominal)
    jes_down x sf_up   : joint= 1677.590460  1D-sum= 1677.590460  error=  +0.000000  (+0.0000% of nominal)
    jes_down x sf_down : joint= 1269.285927  1D-sum= 1269.285928  error=  -0.0

## Level 13 — the same program on real data, under a process pool

Two things at once.

**Propagation crosses a process boundary.** `gak.apply_correction` used to record an External node
wrapping a local closure, which the local `ProcessPoolExecutor` could not ship. That is fixed: the
plan below pickles and runs on a persistent 4-worker pool. Propagation is now the plain path —
no hand fan-out.

**The discriminating control.** The second run registers the *same six tags* with `points=`
omitted. Those four joint-looking labels then carry the default points `{sf: jesup_up}` etc., which
project to `{}` on both the value and the scale-factor containers, so they collapse onto the
one-at-a-time b-tag universes and are numerically identical to them. The "factorization error"
would then read a pure `jes` artifact. Nothing warns. Always check `graphed.points()` before
trusting a joint number.

In [14]:
import uproot

import graphed_histogram as gh
import hist.graphed as hg
from graphed_executors.local import ProcessPoolExecutor


def real_program(with_points):
    g = uproot.graphed("data/Run2012B_SingleMu_50k.root:Events", library="ak")
    raw = g.Jet_pt
    jes_pt = graphed.vary(raw, "jes", up=raw * 1.05, down=raw * 0.95)
    keep = jes_pt > 30.0
    passes = gak.sum(keep, axis=1) >= 2

    def sf(systematic):
        per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                       args=[systematic, "$0"])
        return gak.prod(per_jet[keep], axis=1) * passes

    central, up, down = sf("central"), sf("up"), sf("down")
    points = {"jesup_up":   {"jes": "up",   "sf": "up"},
              "jesup_down": {"jes": "up",   "sf": "down"},
              "jesdn_up":   {"jes": "down", "sf": "up"},
              "jesdn_down": {"jes": "down", "sf": "down"}}
    weight = graphed.vary(central, "sf",
        variations={"up": up, "down": down, "jesup_up": up, "jesup_down": down,
                    "jesdn_up": up, "jesdn_down": down},
        **({"points": points} if with_points else {}))

    h = hg.Hist.new.Reg(1, 0.0, 1e9, name="ht").Double().fill(
        ht=gak.sum(jes_pt[keep], axis=1), weight=[weight])
    return gh.plan({"h": h}, steps_per_file=8, backend="graphed.awkward:AwkwardBackend"), weight


executor = ProcessPoolExecutor(max_workers=4, persistent=True)
try:
    plan, weight = real_program(True)
    res = executor.run(plan)
    tot = {k: float(v.sum()) for k, v in gh.unpack(res.value)["h"].items()}
    print("ProcessPoolExecutor: OK -", res.n_partitions, "partitions, ",
          len(tot), "universes; apply_correction survived the pool")
    print("points['sf_jesup_up'] =", graphed.points(weight)["sf_jesup_up"])
    for k, v in tot.items():
        print(f"    {k:14s} {v:14.6f}")
    real = factorization_error("\nfactorization error on the skim", tot)

    plan_c, weight_c = real_program(False)      # ... the same six tags, points= OMITTED
    tot_c = {k: float(v.sum()) for k, v in gh.unpack(executor.run(plan_c).value)["h"].items()}
    print("CONTROL, points= omitted: points['sf_jesup_up'] =",
          graphed.points(weight_c)["sf_jesup_up"])
    print(f"    {'sf_up':14s} {tot_c['sf_up']:14.6f}")
    print(f"    {'sf_jesup_up':14s} {tot_c['sf_jesup_up']:14.6f}"
          "   <- identical: the joint label collapsed onto the 1-D one")
    bad = tot_c["sf_jesup_up"] - (tot_c["jes_up"] + tot_c["sf_up"] - tot_c["nominal"])
    good = tot["sf_jesup_up"] - (tot["jes_up"] + tot["sf_up"] - tot["nominal"])
    print(f"    would-be 'error' {bad:+.6f}  vs the real {good:+.6f}"
          "   <- a pure jes artifact, silently")
finally:
    executor.close()

ProcessPoolExecutor: OK - 8 partitions,  9 universes; apply_correction survived the pool
points['sf_jesup_up'] = {'jes': 'up', 'sf': 'up'}
    nominal           9233.000000
    jes_up            9950.000000
    jes_down          8513.000000
    sf_up            11237.497496
    sf_down           7507.938209
    sf_jesup_up      12156.013417
    sf_jesup_down     8059.085588
    sf_jesdn_up      10321.268887
    sf_jesdn_down     6949.713250

factorization error on the skim
    jes_up   x sf_up   : joint=12156.013417  1D-sum=11954.497496  error=+201.515921  (+2.1826% of nominal)
    jes_up   x sf_down : joint= 8059.085588  1D-sum= 8224.938209  error=-165.852621  (-1.7963% of nominal)
    jes_down x sf_up   : joint=10321.268887  1D-sum=10517.497496  error=-196.228609  (-2.1253% of nominal)
    jes_down x sf_down : joint= 6949.713250  1D-sum= 6787.938209  error=+161.775041  (+1.7521% of nominal)
    -> largest |error| = 2.1826% of nominal



CONTROL, points= omitted: points['sf_jesup_up'] = {'sf': 'jesup_up'}
    sf_up            11237.497496
    sf_jesup_up      11237.497496   <- identical: the joint label collapsed onto the 1-D one
    would-be 'error' -717.000000  vs the real +201.515921   <- a pure jes artifact, silently


## Level 14 — `points=` is not weight-only: a joint SHIFT ⊗ SHIFT universe

Registration resolves by projection too, so a joint *kinematic* universe (JES ⊗ JER) is expressible
with `collections=`. The inner `jes` universe the point names is kept through registration rather
than flattened to nominal.

In [15]:
s, c = ctx0()
pt, eta = c["pt"], c["eta"]

a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
b = graphed.vary(a, "jer", collections={"eta": {"up": a["eta"] * 1.01, "down": a["eta"] * 0.99}})
jj = graphed.vary(b, "jesjer",
                  collections={"pt": {"upup": b["pt"] * 1.15, "dndn": b["pt"] * 0.85}},
                  points={"upup": {"jes": "up",   "jer": "up"},
                          "dndn": {"jes": "down", "jer": "down"}})
show(jj)

labels    : ('nominal', 'jes_up', 'jes_down', 'jesjer_upup', 'jesjer_dndn', 'jer_up', 'jer_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    jesjer_upup            {'jer': 'up', 'jes': 'up'}
    jesjer_dndn            {'jer': 'down', 'jes': 'down'}
    jer_up                 {'jer': 'up'}
    jer_down               {'jer': 'down'}


## Level 15 — renormalization ⊗ factorization scale: the 7-point grid

The first of the two motivating real cases. `mu_R` and `mu_F` are two independent axes, and the
standard prescription varies them over a **named set of 7 points** in that 2-D space:

```
(mu_R, mu_F) in {(1,1), (1/2,1), (2,1), (1,1/2), (1,2), (1/2,1/2), (2,2)}
```

The two "extreme" corners `(1/2, 2)` and `(2, 1/2)` are deliberately **excluded** — that is the
whole reason this is a *set of points* and not a grid you can generate: it is neither the union of
the one-at-a-time set nor the full cross product.

The four axis-aligned points are ordinary tags. The two diagonal points sit on **two axes at once**
and need `points=`. Note the coordinates are typed as **numbers** (`0.5`, `2`) and reach families
registered with the numeric tags `"0p5"` / `"2"`: coordinates are compared by *value*, so `0.5`,
`"0.5"`, `"0p5"` and `Fraction(1,2)` are one coordinate.

7 universes, not 9 (the excluded corners) and not 5 (the one-at-a-time set alone).

In [16]:
s, c = ctx0()
w0 = c["pt"] * 0.0 + 1.0                # a flat per-event weight to scale

scale = graphed.vary(w0, "muR", variations={"0p5": w0 * 0.92, "2": w0 * 1.07})
scale = graphed.vary(scale, "muF", variations={"0p5": w0 * 0.95, "2": w0 * 1.04})
scale = graphed.vary(scale, "scale",
                     variations={"dndn": w0 * 0.87, "upup": w0 * 1.12},
                     points={"dndn": {"muR": 0.5, "muF": 0.5},     # numbers, not strings
                             "upup": {"muR": 2,   "muF": 2}})
show(scale)
print()
print(len(graphed.labels(scale)), "universes: 1 nominal + 4 axis-aligned + 2 diagonal.")
print("the label keeps YOUR spelling ('muR_0p5'); the point renders the VALUE ('5em1'),")
print("which is what predicts resolution.")

labels    : ('nominal', 'muR_0p5', 'muR_2', 'muF_0p5', 'muF_2', 'scale_dndn', 'scale_upup')
points    :
    nominal                {}
    muR_0p5                {'muR': '5em1'}
    muR_2                  {'muR': '2'}
    muF_0p5                {'muF': '5em1'}
    muF_2                  {'muF': '2'}
    scale_dndn             {'muF': '5em1', 'muR': '5em1'}
    scale_upup             {'muF': '2', 'muR': '2'}

7 universes: 1 nominal + 4 axis-aligned + 2 diagonal.
the label keeps YOUR spelling ('muR_0p5'); the point renders the VALUE ('5em1'),
which is what predicts resolution.


## Level 16 — a PDF eigenvector set

The second motivating case, and the one where it pays to be precise about what needs `points=`.

**A Hessian eigenvector set does NOT need `points=`.** Each eigenvector is its own independent
nuisance with its own ±1σ members, so N eigenvectors are N families of two tags each: `2N + 1`
universes, composed as the union. Registering them as one family with many tags would be *wrong* —
it would tell the fit they are one parameter.

**What does need `points=`** is any member that is a **direction** in that space, i.e. displaced on
more than one axis at once:

- the **PDF ⊗ α_s** combined member — the standard prescription pairs an α_s displacement with a
  PDF one, so it sits at `{pdf1: +1, alphaS: +1}`;
- a **replica** — an MC replica is a sample of the joint PDF distribution and sits at a coordinate
  on *every* eigen-axis at once (three, below).

The fit's answer to a multi-axis direction is reparameterization: declare the direction as a new
named nuisance. `points=` supplies exactly that — the family name **is** the reparameterized
nuisance, and the point is the coordinate map recorded alongside it.

Coordinates must be **registered tags** of their nuisance, so a replica can only sit at coordinates
that were declared; that check is what stops a replica point from silently resolving to nominal.

In [17]:
s, c = ctx0()
w0 = c["pt"] * 0.0 + 1.0

pdf = w0
for i in (1, 2, 3):                      # three Hessian eigenvectors, +-1 sigma each
    pdf = graphed.vary(pdf, f"pdf{i}",
                       variations={"1": w0 * (1 + 0.01 * i), "-1": w0 * (1 - 0.01 * i)})
print("the eigenvector set alone -- no points= anywhere:")
print("   ", graphed.labels(pdf), f"= 2*3 + 1 = {len(graphed.labels(pdf))} universes")

alphas = graphed.vary(pdf, "alphaS", variations={"1": w0 * 1.015, "-1": w0 * 0.985})

combined = graphed.vary(alphas, "pdfas",
                        variations={"up": w0 * 1.026, "down": w0 * 0.974},
                        points={"up":   {"pdf1":  1, "alphaS":  1},
                                "down": {"pdf1": -1, "alphaS": -1}})

replicas = graphed.vary(combined, "replica",
                        variations={"1": w0 * 1.008, "2": w0 * 0.994},
                        points={"1": {"pdf1":  1, "pdf2": -1, "pdf3":  1},
                                "2": {"pdf1": -1, "pdf2":  1, "pdf3": -1}})

print()
show(replicas)
print()
print("the members that NEEDED points= (>=2 coordinates):")
for label, point in graphed.points(replicas).items():
    if len(point) > 1:
        print(f"    {label:14s} {point}")

the eigenvector set alone -- no points= anywhere:
    ('nominal', 'pdf1_1', 'pdf1_m1', 'pdf2_1', 'pdf2_m1', 'pdf3_1', 'pdf3_m1') = 2*3 + 1 = 7 universes

labels    : ('nominal', 'pdf1_1', 'pdf1_m1', 'pdf2_1', 'pdf2_m1', 'pdf3_1', 'pdf3_m1', 'alphaS_1', 'alphaS_m1', 'pdfas_up', 'pdfas_down', 'replica_1', 'replica_2')
points    :
    nominal                {}
    pdf1_1                 {'pdf1': '1'}
    pdf1_m1                {'pdf1': 'm1'}
    pdf2_1                 {'pdf2': '1'}
    pdf2_m1                {'pdf2': 'm1'}
    pdf3_1                 {'pdf3': '1'}
    pdf3_m1                {'pdf3': 'm1'}
    alphaS_1               {'alphaS': '1'}
    alphaS_m1              {'alphaS': 'm1'}
    pdfas_up               {'alphaS': '1', 'pdf1': '1'}
    pdfas_down             {'alphaS': 'm1', 'pdf1': 'm1'}
    replica_1              {'pdf1': '1', 'pdf2': 'm1', 'pdf3': '1'}
    replica_2              {'pdf1': 'm1', 'pdf2': '1', 'pdf3': 'm1'}

the members that NEEDED points= (>=2 coordinates):
 

## Level 17 — numeric coordinates are canonicalized by VALUE

Tags may be numeric strings, including the datacard `p`-form. The **label keeps your spelling**;
`graphed.points()` renders the **value**. Two spellings of one value are therefore one universe,
and registering both is refused: two labels for one point would mean two histogram bins and two
content hashes.

In [18]:
s, c = ctx0()
pt = c["pt"]

scan = graphed.vary(pt, "morph", **{"0p5": pt * 1.05})
scan = graphed.vary(scan, "morph", variations={"1": pt * 1.10, "2.0": pt * 1.20, "-1": pt * 0.90})
show(scan)

print()
try:
    graphed.vary(scan, "morph", variations={"0.5": pt * 9.99})   # "0.5" is "0p5" by value
except graphed.GraphedError as e:
    print("refused:", e)

labels    : ('nominal', 'morph_0p5', 'morph_1', 'morph_2', 'morph_m1')
points    :
    nominal                {}
    morph_0p5              {'morph': '5em1'}
    morph_1                {'morph': '1'}
    morph_2                {'morph': '2'}
    morph_m1               {'morph': 'm1'}

refused: variation tags '0p5' and '5em1' in family 'morph' name the same value (1/2); two labels for one universe would mean two bins and two content hashes


## Level 18 — the requested spelling, typed as numbers

The original request was `add_variation(point={"nuisance": 1.0, "correlated_nuisance": -1.0},
value=...)`. That is an HS3 *parameter point*, and it is accepted **as typed** against families
registered with numeric tags — `1` canonicalizes to `"1"`, `-1` to `"m1"`.

Identifier tags are **not** promoted to ±1: `up` stays an opaque coordinate on its axis, so a point
written with numbers reaches only numerically-tagged families. That is enforced, not silent
(level 20).

In [19]:
s, c = ctx0()
pt = c["pt"]

sh = graphed.vary(c, "jes", collections={"pt": {"1": pt * 1.1, "-1": pt * 0.9}})
wv = sh["pt"] * 0.5
num = graphed.vary(sh, "btag", wv, is_weight=True, variations={"1": wv * 1.2, "-1": wv * 0.8})
ambient = graphed.weight(num)

r2 = graphed.vary(num, "jesbtag_corr", ambient, is_weight=True,
                  variations={"up": ambient * 1.37},
                  points={"up": {"jes": 1, "btag": -1}})     # the request, verbatim

print("labels                    :", graphed.labels(r2))
print("points['jesbtag_corr_up'] :", graphed.points(r2)["jesbtag_corr_up"])
print("variations()['jes']       :", graphed.variations(r2)["jes"])

labels                    : ('nominal', 'jes_1', 'jes_m1', 'btag_1', 'btag_m1', 'jesbtag_corr_up')
points['jesbtag_corr_up'] : {'btag': 'm1', 'jes': '1'}
variations()['jes']       : {'1': ('shift', Fraction(1, 1)), 'm1': ('shift', Fraction(-1, 1))}


## Level 19 — the zero asymmetry

A **tag** `"0"` and a **coordinate** `0` are different things, deliberately.

- A registered tag `"0"` is a *name* for a universe, so it mints the ordinary label `shift_0` with
  point `{shift: 0}` — a real universe, distinct from nominal.
- A **coordinate** `0` inside a `points=` map means "this axis sits at its central value", which is
  what absence already says. It is **dropped**, so `{jes: 1, btag: 0}` reduces to `{jes: 1}` — which
  the default label `jes_1` already names, so the entry is refused as a second name for one universe.
- A `points=` entry that reduces to the empty point names the origin, and is refused outright.

In [20]:
zero_tag = graphed.vary(num, "shift", ambient, is_weight=True, variations={"0": ambient * 1.01})
print("a TAG '0' mints a real universe:",
      graphed.labels(zero_tag)[-1], graphed.points(zero_tag)["shift_0"])

print()
try:                                          # a zero COORDINATE is dropped first
    graphed.vary(num, "jz", ambient, is_weight=True, variations={"a": ambient * 1.5},
                 points={"a": {"jes": 1, "btag": 0}})
except graphed.GraphedError as e:
    print("zero coordinate dropped ->", e)

print()
try:
    graphed.vary(num, "orig", ambient, is_weight=True, variations={"x": ambient * 1.5},
                 points={"x": {}})
except graphed.GraphedError as e:
    print("the origin ->", e)

a TAG '0' mints a real universe: shift_0 {'shift': '0'}

zero coordinate dropped -> point {'jes': '1'} is already registered under label 'jes_1', so label 'jz_a' would be a second name for one universe — two slots, two StrCategory bins and two content hashes

the origin -> points= entry 'x' names the central universe — every coordinate sits at 0, which is what absence already says; nominal is not a variation


## Level 20 — the construction-time refusals

`points=` earns a new label only for a **≥2-coordinate** universe. Everything one-at-a-time is
spelled as a plain tag. Five refusals enforce that, all fired below:

1. **single coordinate** — the default label already names that point.
2. **unreachable coordinate** — a number typed against identifier-tagged families, naming what *is*
   registered. This is what stops the requested numeric spelling from silently returning nominal.
3. **unknown key** — a `points=` key that is not a tag of this call.
4. **one label, one point** — Session-scoped, across *independent* containers.
5. (level 19) the origin entry, and the zero-coordinate reduction.

Inherited labels keep the silent nominal fallback — partial coverage across containers is a
legitimate pattern. A *typed* coordinate is not.

In [21]:
s, c = ctx0()
pt = c["pt"]
sh = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
wv = sh["pt"] * 0.5
base = graphed.vary(sh, "btag", wv, is_weight=True, variations={"up": wv * 1.2, "down": wv * 0.8})
amb = graphed.weight(base)

for why, pts in (("single coordinate    ", {"x": {"jes": "up"}}),
                 ("unreachable coordinate", {"x": {"jes": 1, "btag": -1}}),
                 ("unknown key          ", {"y": {"jes": "up", "btag": "down"}})):
    try:
        graphed.vary(base, "probe", amb, is_weight=True, variations={"x": amb * 1.5}, points=pts)
    except graphed.GraphedError as e:
        print(f"{why} -> {e}\n")

# 4. one label, one point -- two INDEPENDENT containers in ONE Session
s2 = Session(NumpyBackend())


def container(name):
    r = from_record(s2, name, pt=np.arange(1.0, 13.0))
    ctx = EventContext(s2, r["pt"], collections={"pt": r["pt"]})
    a = graphed.vary(ctx, "jes", collections={"pt": {"up": ctx["pt"] * 1.1,
                                                     "down": ctx["pt"] * 0.9}})
    w = a["pt"] * 0.5
    b = graphed.vary(a, "btag", w, is_weight=True, variations={"up": w * 1.2, "down": w * 0.8})
    return b, graphed.weight(b)


b1, a1 = container("ev1")
j1 = graphed.vary(b1, "jesbtag", a1, is_weight=True, variations={"upup": a1 * 1.37},
                  points={"upup": {"jes": "up", "btag": "up"}})
print("container 1 registered:", graphed.labels(j1)[-1], graphed.points(j1)["jesbtag_upup"])

b2, a2 = container("ev2")
try:
    graphed.vary(b2, "jesbtag", a2, is_weight=True, variations={"upup": a2 * 1.37},
                 points={"upup": {"jes": "down", "btag": "down"}})
except graphed.GraphedError as e:
    print("one label, one point  ->", e)

single coordinate     -> point {'jes': 'up'} is already registered under label 'jes_up', so label 'probe_x' would be a second name for one universe — two slots, two StrCategory bins and two content hashes

unreachable coordinate -> points= on graphed.vary('probe'): 'm1' is not a registered tag of nuisance 'btag', whose tags are ['down', 'up']

unknown key           -> points= key 'y' is not a tag of this graphed.vary('probe') call, whose tags are ['x']



container 1 registered: jesbtag_upup {'btag': 'up', 'jes': 'up'}


one label, one point  -> variation label 'jesbtag_upup' already names the point {'btag': 'up', 'jes': 'up'} in this Session, and this call names {'btag': 'down', 'jes': 'down'}; one label names one universe


## Where each level lands

| # | Level | What it adds |
|---|---|---|
| 1 | loose up/down | `f"{name}_{tag}"`, default point `{name: tag}` |
| 2 | extending a family | a second call, same nuisance — still one fit parameter |
| 3 | weight (`is_weight=True`) | kind `'weight'`; selection fixed |
| 4 | shift (`collections=`) | kind `'shift'`; selection moves |
| 5 | lockstep | one nuisance, several collections, no extra universes |
| 6 | stacked families | composition is the **union**: 2+2 → 5, not 9 |
| 7 | shift then weight | the ambient weight carries inherited labels |
| 8 | **name identity** | one name, both effects, one universe; kind `'both'` |
| 9 | **propagation** | `gak.apply_correction` over a `Varied`, one node per universe |
| 10 | **`points=`** | a universe on ≥2 axes; same objects, different points |
| 11 | projection | how a joint label resolves on a container that knows one axis |
| 12 | factorization error | what a joint universe measures, with a zero control |
| 13 | real data + process pool | propagation crosses the pool; the silent-collapse control |
| 14 | shift ⊗ shift | `points=` is not weight-only |
| 15 | mu_R ⊗ mu_F | a named 7-point set: neither union nor cross product |
| 16 | PDF eigenvectors | the set needs no `points=`; a *direction* in it does |
| 17 | numeric coordinates | canonicalized by value; label keeps the spelling |
| 18 | the requested spelling | numbers accepted as typed |
| 19 | zero asymmetry | tag `"0"` is a universe; coordinate `0` is absence |
| 20 | refusals | `points=` earns a label only for ≥2 coordinates |

## Traps worth carrying away

- **`graphed.variations()` is context-only.** On a bare `Varied` it raises. `graphed.points()`
  reads both, and raises on executed results — a label cannot be parsed back into a point, and
  points are a record-time fact that is not carried on disk.
- **Joint members must be the same expression objects** as the one-at-a-time members. An
  arithmetically equal but distinct expression adds a node the optimizer may merge back, which the
  histogram builders refuse at compile time.
- **A single-coordinate `points=` entry is always refused.** One-at-a-time universes are plain tags.
- **Numbers reach only numerically-tagged families.** `up` is not +1σ at this layer; that is a
  stats-export convention, not a frontend one.
- **Enumerating joint-looking tags without `points=` is the silent wrong number** (level 13). Check
  `graphed.points()` before trusting a joint number.